# RAG Step 3: Retrieve Relevant Context

Search the persistent ChromaDB collection for chunks that are semantically similar to a user question. The returned passages, metadata, and distances are inputs to the generation step. Run Step 2 first so the collection has records.


In [ ]:
# Install once if needed: %pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
 )
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(
    name="rag_demo_collection", embedding_function=embedding_fn
 )

question = "How can RAG reduce unsupported answers?"
available = collection.count()
if available == 0:
    raise RuntimeError("The collection is empty. Run RAG Step 2 first.")

results = collection.query(
    query_texts=[question],
    n_results=min(3, available),
    include=["documents", "metadatas", "distances"],
 )

matches = []
for record_id, text, metadata, distance in zip(
    results["ids"][0], results["documents"][0],
    results["metadatas"][0], results["distances"][0]
 ):
    matches.append({"id": record_id, "text": text, "metadata": metadata, "distance": distance})

context = "\n\n".join(item["text"] for item in matches)
print("Question:", question)
for item in matches:
    print(f\"\n{item['id']} | distance={item['distance']:.4f} | {item['metadata']}\")
    print(item["text"])
print("\nContext for the LLM:\n", context)


## Interpret results

Smaller distances indicate closer neighbors for this collection's configured metric; a distance is not a probability or a guarantee of relevance. Check retrieved passages and source metadata. Tune top-k and any relevance cutoff against representative questions. Continue with Step 4 to generate a context-grounded answer.